# Imports and setup

### Auto-re-import python modules, useful for editing local files

In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [2]:
import itertools
import logging
import os
import warnings
from collections import Counter

import matplotlib as mpl
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)

# pl.Config.set_verbose()
# mpl.rcParams["figure.max_open_warning"] = 0

from notifications import notify, notify_done
from scop_constants import n_scop_cols, same_scop_cols
from sensitivity_until_first_false_positive_mem_optimized import (
    MultisearchSensitivityCalculator,
)
from sourmash_constants import sourmash_score_cols

pl.Config.set_verbose(False)

INFO:aiobotocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


polars.config.Config

In [3]:
pl.__version__

'1.9.0'

# Read in data

In [4]:
! aws s3 sync --exclude '*' --include '*hp.k1*.filtered.pq' \
    --dryrun \
    s3://seanome-kmerseek/scope-benchmark/analysis-outputs/hp \
    /home/ec2-user/data/seanome-kmerseek/scope-benchmark/analysis-outputs/hp

In [5]:
! aws s3 sync --exclude '*' --include '*hp.k1*.filtered.pq' \
    s3://seanome-kmerseek/scope-benchmark/analysis-outputs/hp \
    /home/ec2-user/data/seanome-kmerseek/scope-benchmark/analysis-outputs/hp

## Iterate over all ksizes and moltypes

In [6]:
df = (
    pl.scan_parquet(
        "/home/ec2-user/data/seanome-kmerseek/scope-benchmark/analysis-outputs/hp/00_cleaned_multisearch_results/scope40.multisearch.hp.k14.filtered.pq"
    )
    .head()
    .collect()
)
df

query_name,query_md5,match_name,match_md5,containment,max_containment,jaccard,intersect_hashes,prob_overlap,prob_overlap_adjusted,containment_adjusted,containment_adjusted_log10,tf_idf_score,query_family,query_superfamily,query_fold,query_class,n_family,n_superfamily,n_fold,n_class,query_scop_id,match_family,match_superfamily,match_fold,match_class,match_scop_id,same_family,same_superfamily,same_fold,same_class,ksize,moltype,log10_prob_overlap_adjusted,log10_containment,log10_max_containment,log10_tf_idf_score,log10_jaccard
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,cat,cat,cat,cat,i64,i64,i64,i64,str,cat,cat,cat,cat,str,bool,bool,bool,bool,i32,str,f64,f64,f64,f64,f64
"""d2gkma_ a.1.1.1 (A:) Protozoan…","""e3716958c80ece6dc38f8219bff155…","""d5t9pa_ d.58.7.0 (A:) automate…","""1525bdb9db70d519fa350147910028…",0.017544,0.02381,0.010204,2.0,1.0232e-8,2.356778,0.007444,-2.128194,0.094676,"""a.1.1.1""","""a.1.1""","""a.1""","""a""",5,57,62,2644,"""d2gkma_""","""d.58.7.0""","""d.58.7""","""d.58""","""d""","""d5t9pa_""",false,false,false,false,14,"""hp""",0.372319,-1.755875,-1.623249,-1.023759,-1.991226
"""d2dc3a_ a.1.1.2 (A:) Cytoglobi…","""c6dec422df6443edf2e01cbeb23317…","""d5t9pa_ d.58.7.0 (A:) automate…","""1525bdb9db70d519fa350147910028…",0.012739,0.02381,0.008368,2.0,1.5373e-8,3.540976,0.003598,-2.443993,0.063981,"""a.1.1.2""","""a.1.1""","""a.1""","""a""",25,57,62,2644,"""d2dc3a_""","""d.58.7.0""","""d.58.7""","""d.58""","""d""","""d5t9pa_""",false,false,false,false,14,"""hp""",0.549123,-1.89487,-1.623249,-1.19395,-2.077368
"""d1ecaa_ a.1.1.2 (A:) Erythrocr…","""00e1b8c192c60e48b02c1a3c4700f0…","""d5t9pa_ d.58.7.0 (A:) automate…","""1525bdb9db70d519fa350147910028…",0.01626,0.02381,0.009756,2.0,1.0788e-8,2.484818,0.006544,-2.18417,0.087363,"""a.1.1.2""","""a.1.1""","""a.1""","""a""",25,57,62,2644,"""d1ecaa_""","""d.58.7.0""","""d.58.7""","""d.58""","""d""","""d5t9pa_""",false,false,false,false,14,"""hp""",0.395295,-1.788875,-1.623249,-1.058674,-2.010724
"""d1hlba_ a.1.1.2 (A:) Hemoglobi…","""79c00d3e55b314fc2fc153a0de1af1…","""d5t9pa_ d.58.7.0 (A:) automate…","""1525bdb9db70d519fa350147910028…",0.020833,0.035714,0.013333,3.0,1.4853e-8,3.421269,0.006089,-2.215428,0.113863,"""a.1.1.2""","""a.1.1""","""a.1""","""a""",25,57,62,2644,"""d1hlba_""","""d.58.7.0""","""d.58.7""","""d.58""","""d""","""d5t9pa_""",false,false,false,false,14,"""hp""",0.534187,-1.681241,-1.447158,-0.943616,-1.875061
"""d3l0fa_ a.1.1.3 (A:) Phycocyan…","""3e99903d904b09029f5ce98c6c4a4e…","""d5t9pa_ d.58.7.0 (A:) automate…","""1525bdb9db70d519fa350147910028…",0.04698,0.083333,0.030973,7.0,4.7374e-8,10.912101,0.004305,-2.365997,0.247737,"""a.1.1.3""","""a.1.1""","""a.1""","""a""",4,57,62,2644,"""d3l0fa_""","""d.58.7.0""","""d.58.7""","""d.58""","""d""","""d5t9pa_""",false,false,false,false,14,"""hp""",1.037908,-1.328088,-1.079181,-0.606009,-1.50901


In [7]:
# df.schema

In [8]:
from sourmash_constants import sourmash_score_cols

In [9]:
sourmash_score_cols

('containment',
 'tf_idf_score',
 'containment_adjusted_log10',
 'log10_prob_overlap_adjusted',
 'log10_max_containment',
 'log10_jaccard',
 'log10_tf_idf_score',
 'intersect_hashes')

In [10]:
# If descending=True, then bigger=better for that score
# if descending=False, then smaller=better (e.g. want a small probability of overlap)

sourmash_score_cols_to_descending = {
    "containment": True,
    "tf_idf_score": True,
    "containment_adjusted_log10": True,
    "log10_prob_overlap_adjusted": False,
    "log10_max_containment": True,
    "log10_jaccard": True,
    "log10_tf_idf_score": True,
    "intersect_hashes": True,
}

In [ ]:

moltype_info = {
    # "protein": dict(
    #     ksizes=range(5, 21),
    #     pipeline_outdir="s3://seanome-kmerseek/scope-benchmark/pipeline-outputs/2024-10-08__protein_k5-20",
    #     analysis_outdir="s3://seanome-kmerseek/scope-benchmark/analysis-outputs/protein",
    # ),
    # "dayhoff": dict(
    #     ksizes=range(5, 21),
    #     pipeline_outdir="s3://seanome-kmerseek/scope-benchmark/pipeline-outputs/2024-10-09__dayhoff_k5-20",
    #     analysis_outdir="s3://seanome-kmerseek/scope-benchmark/analysis-outputs/dayhoff",
    # ),
    "hp": dict(
        ksizes=reversed(range(10, 30)),
        # pipeline_outdir="s3://seanome-kmerseek/scope-benchmark/pipeline-outputs/2024-10-09__hp_k20-60",
        pipeline_outdir="/home/ec2-user/data/seanome-kmerseek/scope-benchmark/pipeline-outputs/2024-10-09__hp_k20-60",
        # analysis_outdir="s3://seanome-kmerseek/scope-benchmark/analysis-outputs/hp",
        analysis_outdir="/home/ec2-user/data/seanome-kmerseek/scope-benchmark/analysis-outputs/hp",
    ),
}

basename_template = r"scope40.multisearch.{moltype}.k{ksize}.filtered.pq"

for moltype, info in moltype_info.items():
    ksizes = info["ksizes"]
    cleaned_outdir = os.path.join(
        info["analysis_outdir"], "00_cleaned_multisearch_results"
    )
    sensitivity_outdir = os.path.join(
        info["analysis_outdir"], "01_sensitivity_until_first_false_positive"
    )
    if not os.path.exists(sensitivity_outdir):
        ! mkdir -p $sensitivity_outdir
    for ksize in ksizes:

        msc = MultisearchSensitivityCalculator(
            moltype, ksize, cleaned_outdir, sensitivity_outdir, chunk_size=1_000_000
        )
        msc.calculate_sensitivity()

INFO:notifications:Processing /home/ec2-user/data/seanome-kmerseek/scope-benchmark/analysis-outputs/hp/00_cleaned_multisearch_results/scope40.multisearch.hp.k15.filtered.pq with chunk size 1000000
INFO:notifications:/home/ec2-user/data/seanome-kmerseek/scope-benchmark/analysis-outputs/hp/01_sensitivity_until_first_false_positive/scope40.multisearch.hp.15.sensitivity_to_first_fp.pq already exists, skipping
INFO:notifications:Processing /home/ec2-user/data/seanome-kmerseek/scope-benchmark/analysis-outputs/hp/00_cleaned_multisearch_results/scope40.multisearch.hp.k14.filtered.pq with chunk size 1000000
INFO:notifications:/home/ec2-user/data/seanome-kmerseek/scope-benchmark/analysis-outputs/hp/01_sensitivity_until_first_false_positive/scope40.multisearch.hp.14.sensitivity_to_first_fp.pq already exists, skipping
INFO:notifications:Processing /home/ec2-user/data/seanome-kmerseek/scope-benchmark/analysis-outputs/hp/00_cleaned_multisearch_results/scope40.multisearch.hp.k13.filtered.pq with chun

In [ ]:
! aws s3 sync /home/ec2-user/data/seanome-kmerseek/scope-benchmark/analysis-outputs/hp \
    s3://seanome-kmerseek/scope-benchmark/analysis-outputs/hp

In [ ]:
%debug

In [ ]:
# sensitivity.show_graph()

In [14]:
# sensitivity.comm

In [15]:
sensitivity.shape

NameError: name 'sensitivity' is not defined

In [ ]:
# g = sns.catplot(
#     data=sensitivity,
#     col="sourmash_score",
#     hue="ksize",
#     y="variable",
#     x="sensitivity",
#     col_wrap=4, height=3
# )

In [ ]:
# %debug

In [ ]:
cv